# PySpark job: resample and join the largest table

This notebook runs **the same stages as `src/08_spark_job.py`**, one cell at a time, so the output of each stage (and its row count) is saved here. The brief asks for *row counts before and after* every stage: see the last cell.

Re-run everything with `make spark` (it executes this notebook and saves the outputs).

In [1]:
# Load the script as a module, so the notebook and the script can never drift apart.
import sys, importlib.util
sys.path.insert(0, '../src')
spec = importlib.util.spec_from_file_location('job', '../src/08_spark_job.py')
job = importlib.util.module_from_spec(spec)
spec.loader.exec_module(job)

job.reset_cleaning_log_for_stage(job.STAGE)  # avoid duplicate log rows when both script and notebook run
spark = job.start_spark()
print('Spark', spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/26 19:22:57 WARN Utils: Your hostname, manshu05, resolves to a loopback address: 127.0.1.1; using 10.71.108.100 instead (on interface wlan0)
26/11/26 19:22:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/punya/Downloads/eight-pm-problem/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/11/26 19:22:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.3.0


## Stage 1: load
The full grid from `07_gaps_outliers.py`: every station × parameter × hour, including empty hours.

In [2]:
readings = job.stage_load(spark)
readings.printSchema()
readings.groupBy('qc_flag').count().orderBy('count', ascending=False).show()

  [1_load] Load readings_clean (full station x parameter x hour grid): 279,552 rows (1.9s)
root
 |-- station_id: string (nullable = true)
 |-- ts_ist: timestamp_ntz (nullable = true)
 |-- parameter: string (nullable = true)
 |-- value: double (nullable = true)
 |-- value_raw: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- qc_flag: string (nullable = true)
 |-- is_filled: boolean (nullable = true)
 |-- observed_count: long (nullable = true)



+-------------------+------+
|            qc_flag| count|
+-------------------+------+
|                 ok|200788|
|            missing| 75136|
|           flatline|  1720|
|               zero|   877|
|extreme_unexplained|   785|
|real_event_festival|   145|
|real_event_citywide|    65|
|   failure_reviewed|    21|
|              spike|    10|
|real_event_reviewed|     5|
+-------------------+------+



## Stage 2: clean
Drop hours with no value and every sensor-failure flag. Real extreme events (`real_event_*`) stay in.

In [3]:
clean = job.stage_clean(readings)

  [2_clean] Drop missing hours and QC failures (real events kept): 213,940 rows (0.3s)
  [log #37] 08_spark_job: Rows with no usable value (never sent, or blanked as a sensor failure) -> Filtered out in Spark before aggregating (65612 rows)


## Stage 3: resample to hourly
`date_trunc('hour')` + `avg()`. Our data is already hourly, so the count should not change. That is the check.

In [4]:
hourly = job.stage_resample(clean)
hourly.show(5)

  [3_resample] Resample to hourly (date_trunc + avg) per station x parameter: 213,940 rows (1.0s)


+----------+---------+-------------------+--------------------+----------+---------+
|station_id|parameter|            ts_hour|               value|n_readings|is_filled|
+----------+---------+-------------------+--------------------+----------+---------+
|   BLR5548|       co|2025-10-13 01:00:00|5.418703476482618E-4|         1|        1|
|   BLR5548|       co|2025-10-13 18:00:00|6.106065439672803E-4|         1|        0|
|   BLR5548|       co|2025-10-13 21:00:00|5.384335378323109E-4|         1|        0|
|   BLR5548|       co|2025-10-13 23:00:00|5.498895705521473E-4|         1|        0|
|   BLR5548|       co|2025-10-14 01:00:00|5.785296523517383E-4|         1|        0|
+----------+---------+-------------------+--------------------+----------+---------+
only showing top 5 rows


## Stage 4: join with weather and the calendar
Left joins, so no air-quality hour is ever silently dropped.

In [5]:
joined = job.stage_join(spark, hourly)
joined.select('station_id', 'ts_hour', 'value', 'precip_mm', 'is_rain_hour', 'weekday', 'is_holiday', 'season').show(5)

  [4a_pm25] PM2.5 station-hours only: 55,026 rows (0.3s)


  [4b_join] PM2.5 ⋈ weather_hourly ⋈ calendar (left joins): 55,026 rows (0.6s)


  [log #38] 08_spark_job: Joining PM2.5 hours to weather and calendar -> Left joins keep all 55,026 PM2.5 rows; rows_affected = rows that found no weather (0) or no calendar day (0) (0 rows)


  Check passed: saved timestamps read back with the same IST hour


+----------+-------------------+-----+---------+------------+-------+----------+------------+
|station_id|            ts_hour|value|precip_mm|is_rain_hour|weekday|is_holiday|      season|
+----------+-------------------+-----+---------+------------+-------+----------+------------+
|   BLR5548|2025-10-12 22:00:00| 26.1|      0.0|       false| Sunday|      true|post_monsoon|
|   BLR5548|2025-10-12 23:00:00| 28.5|      0.0|       false| Sunday|      true|post_monsoon|
|   BLR5548|2025-10-13 17:00:00| 29.9|      0.0|       false| Monday|     false|post_monsoon|
|   BLR5548|2025-10-13 21:00:00| 28.7|      0.0|       false| Monday|     false|post_monsoon|
|   BLR5548|2025-10-14 00:00:00| 22.8|     0.05|       false|Tuesday|     false|post_monsoon|
+----------+-------------------+-----+---------+------------+-------+----------+------------+
only showing top 5 rows


## Stage 5: aggregate (hour × weekday, month × hour, rain × hour)

In [6]:
results = job.stage_aggregate(joined)
results['spark_pm25_hour_weekday.csv'].filter('hour >= 17').show(14)

  [5a_hour_weekday] GROUP BY weekday x hour (heatmap): 168 rows (0.2s)


  [5b_month_hour] GROUP BY month x hour: 336 rows (0.2s)


  [5c_rain_hour] GROUP BY rain/dry x hour: 48 rows (0.4s)


+-----------+-------+----+---------+---------------+
|weekday_num|weekday|hour|mean_pm25|n_station_hours|
+-----------+-------+----+---------+---------------+
|          0| Monday|  17|     29.8|            313|
|          0| Monday|  18|     33.2|            323|
|          0| Monday|  19|     37.1|            327|
|          0| Monday|  20|     39.1|            325|
|          0| Monday|  21|     39.8|            319|
|          0| Monday|  22|     40.0|            323|
|          0| Monday|  23|     37.3|            318|
|          1|Tuesday|  17|     30.7|            338|
|          1|Tuesday|  18|     32.6|            333|
|          1|Tuesday|  19|     35.3|            331|
|          1|Tuesday|  20|     38.1|            324|
|          1|Tuesday|  21|     39.2|            321|
|          1|Tuesday|  22|     39.1|            321|
|          1|Tuesday|  23|     37.2|            320|
+-----------+-------+----+---------+---------------+
only showing top 14 rows


## Row counts after every stage

In [7]:
import pandas as pd
row_counts = pd.DataFrame(job.ROW_COUNTS)
row_counts.to_csv(job.TABLES / 'spark_row_counts.csv', index=False)
row_counts

,stage,description,rows,seconds
0,1_load,Load readings_clean (full station x parameter ...,279552,1.9
1,2_clean,Drop missing hours and QC failures (real event...,213940,0.3
2,3_resample,Resample to hourly (date_trunc + avg) per stat...,213940,1.0
3,4a_pm25,PM2.5 station-hours only,55026,0.3
4,4b_join,PM2.5 ⋈ weather_hourly ⋈ calendar (left joins),55026,0.6
5,5a_hour_weekday,GROUP BY weekday x hour (heatmap),168,0.2
6,5b_month_hour,GROUP BY month x hour,336,0.2
7,5c_rain_hour,GROUP BY rain/dry x hour,48,0.4


In [8]:
spark.stop()